In [ ]:
import os
from pathlib import Path

import duckdb
import toml
from loguru import logger

In [ ]:
from horse_logic.utils import set_custom_logger_format

In [ ]:
set_custom_logger_format()

In [ ]:
logger.info(f"Local DuckDB version: {duckdb.__version__}")

In [ ]:
SECRETS_TOML = Path.cwd().parent / ".secrets.toml"

In [ ]:
def load_secrets(secrets_file):
    """
    This function loads secrets from a TOML file. It takes the path to the secrets file as an argument and
    returns a dictionary of secrets if the file exists and can be loaded successfully.
    If the file doesn't exist or an error occurs while loading it, the function logs an error message and returns None.
    """
    if Path(secrets_file).exists():
        try:
            secrets = toml.load(secrets_file)
            logger.info(f"Secrets loaded from {secrets_file}")
            return secrets
        except Exception as e:
            logger.error(f"Failed to load secrets: {e}")
            return None
    else:
        logger.error(f"{secrets_file} not found")

In [ ]:
def create_motherduck_env(secrets):
    if "MD_TOKEN" in secrets.keys():
        os.environ["motherduck_token"] = secrets["MD_TOKEN"]
        logger.info(f'MotherDuck environment variable created: motherduck_token - {os.environ["motherduck_token"][:7]}...')
        return True
    else:
        logger.error("Unable to set MotherDuck environment variable - check for MD_TOKEN in secrets file.")
        return False

In [ ]:
def connect_duckdb_md_token(md_token=None):
    if md_token:
        con = duckdb.connect("md:", config={ 'motherduck_token': md_token})
        return con
    else:
        secrets = load_secrets(SECRETS_TOML)
        if create_motherduck_env(secrets):
            con = duckdb.connect("md:")
            return con
        else:
            return None

In [ ]:
con = connect_duckdb_md_token()

In [ ]:
def get_duckdb_extension_version(con, extension_name=None):
    if extension_name:
        try:
            version_query = f"SELECT extension_name, extension_version FROM duckdb_extensions() WHERE extension_name='{extension_name.lower()}';"
            version_result = con.execute(version_query).df()
            # extension_name_sql = version_result.iloc[0]['extension_name']
            extension_version = version_result.iloc[0]['extension_version']
            if extension_version.strip() == "":
                extension_version = "Unknown"
            logger.info(f"Extension: {extension_name}, Version: {extension_version}")
            return f"Extension: {extension_name}, Version: {extension_version}"
        except Exception as e:
            logger.error(f"{e}")
    return None


In [ ]:
get_duckdb_extension_version(con, "MOTHERDUCK")

In [ ]:
def check_local_duckdb_database():
    """
    Checks if a local DuckDB database exists. It returns the path to the database if it exists, and None otherwise.
    """
    DATA_DIR = Path("../data")
    EXPERIMENT_TYPE = "RPE"  # or CB
    assert DATA_DIR.exists()
    local_ddb = DATA_DIR / f"Experiments_{EXPERIMENT_TYPE}_2023-Q4.ddb"  # DuckDB database name

    logger.info(f"Database file: {local_ddb.resolve()}")

    if local_ddb.exists():
        return local_ddb
    else:
        return None

In [ ]:
local_ddb = check_local_duckdb_database()

In [ ]:
local_ddb

In [ ]:
con_local = duckdb.connect(str(local_ddb.resolve()))

In [ ]:
con_local.sql("SHOW ALL TABLES;")

In [ ]:
# TODO: Why is there not data in the experiment database(s)

con_local.sql("SELECT * FROM Experiments")

In [ ]:
con_local.sql("PRAGMA database_list;")

In [ ]:
class EchoSQLDuckDBConnection:
    """
    Class that wraps a DuckDB connection and adds functionality to echo SQL commands.
    It has an `__init__` method that takes a database path and creates a DuckDB connection, and
    a `sql` method that takes an SQL command and an optional `dry_run` argument.
    If `dry_run` is `True`, the `sql` method prints the SQL command instead of executing it.
    The class also has a `__getattr__` method that delegates any other method calls to the internal DuckDB connection.
    """

    def __init__(self, db_path):
        self.conn = duckdb.connect(db_path)
        self.db_path = Path(db_path).stem

    def sql(self, command, dry_run=True, *args, **kwargs):
        if dry_run:
            logger.info(f"Dry run - SQL: {command}")
            return None
        else:
            logger.info(f"{self.db_path} - executing SQL: {command}")
            return self.conn.sql(command, *args, **kwargs)

    # Delegate other methods to self.conn
    def __getattr__(self, attr):
        return getattr(self.conn, attr)

In [ ]:
def standardise_db_name(name_in):
    return name_in.replace("-", "_").lower()

In [ ]:
secrets = load_secrets(SECRETS_TOML)
create_motherduck_env(secrets)  # Ensure the motherduck_token is set

In [ ]:
def copy_local_ddb_to_motherduck(local_ddb, remote_db_name=None, rm_shares=True, dry_run=True):
    """
    Copies a local DuckDB database to the MotherDuck server. It takes the path to the local database,
    an optional name for the new database, and an optional `dry_run` argument.
    If `dry_run` is `True`, the function prints the SQL commands it would execute instead of executing them.
    If `dry_run` is `False`, the function executes the SQL commands to copy the local database to the MotherDuck server.
    """

    if remote_db_name is None:
        remote_db_name = standardise_db_name(Path(local_ddb).stem)

    if rm_shares:
        cascade = "CASCADE"
    else:
        cascade = ""

    local_con = EchoSQLDuckDBConnection(str(local_ddb))  # overloaded .sql() method to echo sql commands
    logger.info(f"Local DB: {local_ddb}")
    local_con.sql("ATTACH 'md:'", dry_run=dry_run)
    local_con.sql(f"DROP DATABASE IF EXISTS {remote_db_name} {cascade};", dry_run=dry_run)
    local_con.sql(f"CREATE OR REPLACE DATABASE {remote_db_name} FROM CURRENT_DATABASE();", dry_run=dry_run)
    #local_con.sql(f"CREATE OR REPLACE DATABASE {remote_db_name} FROM '{local_ddb.stem}';", dry_run=dry_run)


    if not dry_run:
        logger.info(f"Created database {remote_db_name} on https://motherduck.com")

    return remote_db_name, local_con

In [ ]:
db_name, local_con = copy_local_ddb_to_motherduck(local_ddb, remote_db_name=None, dry_run=True)

In [ ]:
db_name, local_con = copy_local_ddb_to_motherduck(local_ddb, remote_db_name=None, dry_run=False)

In [ ]:
local_con.sql("SHOW ALL DATABASES;", dry_run=False)

In [ ]:
local_con.sql("SHOW ALL TABLES;", dry_run=False)

In [ ]:
local_con.sql("PRAGMA md_version;", dry_run=False)

In [ ]:
con_md = duckdb.connect("md:memory:")

In [ ]:
con_md.sql("SHOW ALL DATABASES;")

In [ ]:
con_md.close()

In [ ]:
# Code for working with shared database

# con_md_share = duckdb.connect()

In [ ]:
# -- Run this snippet to attach database
# ATTACH 'md:_share/experiments_rpe_2023_q4/e7b8df3a-fc41-4578-9ce0-553b30685626'


# con_md_share.sql("ATTACH 'md:_share/clouddb/cbe96e98-c6b3-4e13-86b4-c3b42cf95aa6' AS md_ddb")
# con_md_share.sql("USE md_ddb;")

# Learn about sharing here: https://motherduck.com/docs/key-tasks/querying-a-shared-motherduck-database.

In [ ]:
# -- Run this snippet to attach database
# ATTACH 'md:_share/experiments_rpe_2023_q4/e7b8df3a-fc41-4578-9ce0-553b30685626'


# con_md_share.sql("ATTACH 'md:_share/clouddb/cbe96e98-c6b3-4e13-86b4-c3b42cf95aa6' AS md_ddb")
# con_md_share.sql("USE md_ddb;")

# Learn about sharing here: https://motherduck.com/docs/key-tasks/querying-a-shared-motherduck-database.

In [ ]:
# con_md_share.sql("SELECT * FROM Experiments")

In [ ]:
# con_md_share.close()